In [1]:
import sys
sys.path.insert(0, '/Users/pushpita/Documents/ML_Projects/echoagent')
from pathlib import Path
from typing import Dict, List, Any

from backend.agents.prompt_parser import parse_prompt, LLMClient
from backend.agents.vibe_intent import VibeIntent
from backend.agents.planner_agent import PlaylistPlan, PlannerAgent
from backend.agents.relational_retrieval_agent import retrieve_relational_candidates
from backend.agents.vector_retrieval_agent import retrieve_vector_candidates
from backend.agents.candidate_fuser import fuse_candidates
from backend.agents.reranker import rerank_candidates
from backend.agents.playlist_builder import PlaylistBuilderAgent, build_playlist_node

E0000 00:00:1782504493.594437 9919843 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1782504493.594461 9919843 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1782504493.594462 9919843 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1782504493.594464 9919843 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1782504493.594465 9919843 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


In [2]:
# Database configuration
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "backend"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
DATA_DIR = PROJECT_ROOT / "database"
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))

In [4]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

prompt = "Late night chill"

HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN and not HF_TOKEN.startswith("hf_PASTE"), \
    "Add your HF token to .env: HF_TOKEN=hf_..."

MODEL = "Qwen/Qwen2.5-7B-Instruct"

llm_client = LLMClient(
    model_name=MODEL,
    device=None,
    api_key=HF_TOKEN,
    endpoint=None,
    temperature=0.7,
    max_new_tokens=512,
)

parsing = parse_prompt(prompt, prompt_schema_path=f"{SRC_DIR}/agents/prompt_schema.json", llm_client=llm_client)

database_url = f"sqlite:///{DATA_DIR}/music_relational.db"
chroma_persist_directory = f"{DATA_DIR}/chroma_db"

print(f"Relational Database: {database_url}")
print(f"Vector Database: {chroma_persist_directory}")

Relational Database: sqlite:////Users/pushpita/Documents/ML_Projects/echoagent/database/music_relational.db
Vector Database: /Users/pushpita/Documents/ML_Projects/echoagent/database/chroma_db


In [8]:
relational_candidates = retrieve_relational_candidates(
    intent=parsing,
    limit=100,
    database_url=database_url,
)
vector_candidates = retrieve_vector_candidates(
    intent=parsing,
    limit=100,
    persist_directory=chroma_persist_directory,
)

print(f"✓ Relational candidates: {len(relational_candidates)}")
print(f"✓ Vector candidates:    {len(vector_candidates)}")

if relational_candidates:
    print(f"\nRelational sample:{relational_candidates[0].get('title', 'N/A')}")
if vector_candidates:
    print(f"Vector sample:{vector_candidates[0].get('metadata', {}).get('title', 'N/A')}")

✓ Relational candidates: 100
✓ Vector candidates:    100

Relational sample:b'Je Sais Que La Terre Est Plate'
Vector sample:N/A


In [9]:
# Test 1: Fuse relational and vector candidates
if relational_candidates or vector_candidates:
    print("=" * 60)
    print("TEST 1: Candidate Fusion (Relational + Vector)")
    print("=" * 60)

    # Fuse candidates from both sources
    fused_candidates = fuse_candidates(
        relational_candidates=relational_candidates,
        vector_candidates=vector_candidates,
        relational_weight=1.0,
        vector_weight=1.0,
    )

    print(f"✓ Fused {len(fused_candidates)} candidates from both sources")
    print(f"\nFusion breakdown:")
    both = sum(1 for c in fused_candidates if len(c.get('sources', [])) > 1)
    relational_only = sum(1 for c in fused_candidates if c.get('sources') == ['relational'])
    vector_only = sum(1 for c in fused_candidates if c.get('sources') == ['vector'])
    print(f"  Both sources: {both}")
    print(f"  Relational only: {relational_only}")
    print(f"  Vector only: {vector_only}")

    if fused_candidates:
        print(f"\nTop 3 fused candidates:")
        for i, candidate in enumerate(fused_candidates[:3], 1):
            metadata = candidate.get('metadata', {})
            if isinstance(metadata, dict):
                title = metadata.get('title', 'N/A')
            else:
                title = str(metadata)[:50]
            sources = ', '.join(candidate.get('sources', []))
            score = candidate.get('retrieval_score', 0)
            print(f"  {i}. [{sources:15s}] {title} (score: {score:.2f})")
    print("✓ TEST 1 PASSED\n")
else:
    print("⚠️  Skipping TEST 1: No candidates from either source")
    fused_candidates = []

TEST 1: Candidate Fusion (Relational + Vector)
✓ Fused 199 candidates from both sources

Fusion breakdown:
  Both sources: 1
  Relational only: 99
  Vector only: 99

Top 3 fused candidates:
  1. [relational, vector] b'Antarctica Starts Here' (score: 1.03)
  2. [relational     ] b'Je Sais Que La Terre Est Plate' (score: 1.00)
  3. [relational     ] b'On Efface' (score: 1.00)
✓ TEST 1 PASSED



In [10]:
if fused_candidates:
    print("=" * 60)
    print("TEST 2: Reranking Fused Candidates")
    print("=" * 60)

    ranked = rerank_candidates(fused_candidates, parsing, playlist_plan={
        'playlist_size': 20,
        'semantic_weight': 0.25,
        'soft_preference_weight': 0.4,
        'exclusion_penalty': 2.0,
    })

    print(f"✓ Reranked {len(ranked)} candidates")
    print(f"  Score range: {ranked[-1].get('score', 0):.2f} → {ranked[0].get('score', 0):.2f}")

    print(f"\nTop 5 reranked candidates:")
    for i, candidate in enumerate(ranked[:5], 1):
        metadata = candidate.get('metadata', {})
        title = metadata.get('title', 'N/A') if isinstance(metadata, dict) else str(metadata)[:50]
        print(f"  {i}. {title} (score: {candidate.get('score', 0):.2f})")
    print("✓ TEST 2 PASSED\n")
else:
    print("⚠️  Skipping TEST 2: No fused candidates available")
    ranked = []

TEST 2: Reranking Fused Candidates
✓ Reranked 199 candidates
  Score range: 0.12 → 1.14

Top 5 reranked candidates:
  1. b'Antarctica Starts Here' (score: 1.14)
  2. N/A (score: 1.12)
  3. b'Je Sais Que La Terre Est Plate' (score: 1.00)
  4. b'On Efface' (score: 1.00)
  5. b'Howells Delight' (score: 1.00)
✓ TEST 2 PASSED



In [12]:
if ranked and len(ranked) >= 20:
    print("=" * 60)
    print("TEST 3: Build Playlist (20 tracks)")
    print("=" * 60)

    state = {
        "user_prompt": prompt,
        "intent": parsing,
        "playlist_plan": PlaylistPlan(),
        "ranked_candidates": ranked,
        "playlist_builder_agent": PlaylistBuilderAgent(llm_client=llm_client),
    }

    result = build_playlist_node(state)
    playlist = result.get("playlist", [])

    print(f"✓ Generated playlist with {len(playlist)} tracks")
    print(f"\nFinal Playlist:")
    for i, track in enumerate(playlist, 1):
        metadata = track.get('metadata', {})
        title  = metadata.get('title', 'N/A') if isinstance(metadata, dict) else str(metadata)[:40]
        artist = metadata.get('artist_name', 'N/A') if isinstance(metadata, dict) else 'N/A'
        score  = track.get('score', 0)
        print(f"  {i:2d}. {title[:40]:40s} - {artist[:20]:20s} ({score:6.2f})")
    print("✓ TEST 3 PASSED\n")
elif ranked:
    print(f"⚠️  Skipping TEST 3: Only {len(ranked)} ranked candidates (need 20)")
else:
    print("⚠️  Skipping TEST 3: No ranked candidates available")

TEST 3: Build Playlist (20 tracks)
✓ Generated playlist with 20 tracks

Final Playlist:
   1. b'Antarctica Starts Here'                - b'John Cale'         (  1.14)
   2. b'Je Sais Que La Terre Est Plate'        - b'Rapha\xc3\xabl'    (  1.00)
   3. b'On Efface'                             - b'Julie Zenatti'     (  1.00)
   4. b'Howells Delight'                       - b'The Baltimore Cons (  1.00)
   5. b'Martha Served'                         - b'I Hate Sally'      (  1.00)
   6. b'Liquid Time (composition by John Goods - b'Brand X'           (  1.00)
   7. b'Misery Path (From the Privilege of Evi - b'Amorphis'          (  1.00)
   8. b'Nuovi Re pt. I I (feat. Tek money - La - b'Inoki'             (  1.00)
   9. b'Halloween'                             - b'Dead Kennedys'     (  1.00)
  10. b'Parto em terras distantes'             - b'Brigada Victor Jar (  1.00)
  11. b'You Eclipsed By Me (Album Version)'    - b'Atreyu'            (  1.00)
  12. b'Shovel'                            

In [13]:
# Summary
print("=" * 60)
print("✓ PIPELINE TESTS COMPLETED")
print("=" * 60)
print(f"""
Full pipeline tested:
  ✓ Relational database retrieval: {len(relational_candidates)} candidates
  ✓ Vector database retrieval: {len(vector_candidates)} candidates
  ✓ Candidate fusion: {len(fused_candidates)} fused
  ✓ Reranking: {len(ranked)} ranked
  ✓ Playlist building: {'Success' if len(ranked) >= 20 else 'Insufficient data'}

Pipeline tested up to: build_playlist_node()

Tests executed:
  1. Candidate Fusion (Relational + Vector)
  2. Reranking Fused Candidates
  3. Build Playlist (20 tracks)

Next steps would test: critique_node() and full graph execution
""")

✓ PIPELINE TESTS COMPLETED

Full pipeline tested:
  ✓ Relational database retrieval: 100 candidates
  ✓ Vector database retrieval: 100 candidates
  ✓ Candidate fusion: 199 fused
  ✓ Reranking: 199 ranked
  ✓ Playlist building: Success

Pipeline tested up to: build_playlist_node()

Tests executed:
  1. Candidate Fusion (Relational + Vector)
  2. Reranking Fused Candidates
  3. Build Playlist (20 tracks)

Next steps would test: critique_node() and full graph execution

